In [2]:
import requests
import csv

Here's the chunk of code that fetches all the resource IDs for a certain package:

In [ ]:
dataset_id = 'pad_m_nom_sexe'
url = 'https://opendata-ajuntament.barcelona.cat/data/api/action/package_show'

response = requests.get(url, params={'id': dataset_id})
data = response.json()
resources = data['result']['resources']
for res in resources:
    print(f"{res['name']} ({res['id']})")

... and this chunk of code turns the resource id list into a list of tuples, with the year for the dataset coming first:

In [ ]:
id_list = []
for res in resources:
    if res['name'].endswith('.csv'):
        id_list.append((int(res['name'][0:4]), res['id']))
list_length = len(id_list)
print(f'got {list_length} resource IDs')
for i in id_list:
    print(i)
        

... and now write them to a CSV file so i have them saved:

In [ ]:
with open('resource_ids.csv', 'w') as file:
    writer = csv.writer(file)
    for item in id_list:
        writer.writerow(list(item))

Now I start playing with the actual data queries:

In [ ]:
endpoint = "https://opendata-ajuntament.barcelona.cat/data/api/3/action/datastore_search"
resource_id = '12c73de1-c96b-4d0f-8fbc-cc8f63633bde'
response = requests.get(endpoint, params={'resource_id': resource_id, 'limit':5})
data = response.json()


In [ ]:
for i in data['result']['records']:
    print(i)

In [ ]:
def count_rows(resource_id: tuple) -> int:
    endpoint = 'https://opendata-ajuntament.barcelona.cat/data/api/3/action/datastore_search_sql'
    sql = f'SELECT COUNT(*) FROM \"{resource_id[1]}\"'
    params = {'sql': sql}
    response = requests.get(endpoint, params)
    if response.status_code != 200:
        print(f'The row count had a problem: {response.status_code}')
        return None
    else:
        data = response.json()
        row_count = int(data['result']['records'][0]['count'])
        print(f'Counted {row_count} rows in the target record, took {response.elapsed.total_seconds()} seconds to respond.')
        return row_count


In [ ]:
lengths = [count_rows(record) for record in id_list]
max_rows = max(lengths)
print(f'The longest record has {max_rows} rows.')

    

In [ ]:
def get_data(api_endpoint, resource_id: tuple) -> dict:
    rows = count_rows(resource_id)
    if not rows:
        print('The row count went wrong.')
        return {}
    limit = rows + 10
    response = requests.get(api_endpoint, params={
        'resource_id': resource_id[1],
        'limit': limit
    }
                           )
    if response.status_code != 200:
        print(f'Error fetching record: {response.status_code}')
        return {}
    else:
        print(f'API response time: {response.elapsed.total_seconds()} seconds')
        data = response.json()
        return data
    

In [ ]:
resource_tuple = (2014, "f99571fd-2e6d-43f5-9eac-af4542de8aa0")
endpoint = "https://opendata-ajuntament.barcelona.cat/data/api/3/action/datastore_search"
directory = '/home/peter/data_projects/barcelona/people/'
data = get_data(endpoint, resource_tuple)
if data and 'result' in data and 'records' in data['result']:
    count = 0
    with open(f'{directory}names_{resource_tuple[0]}.csv', 'w') as file:
        fieldnames = list(data['result']['records'][0].keys())
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        for i in data['result']['records']:
            writer.writerow(i)
            count += 1
    print(f'Saved a CSV file with {count} entries!')
else:
    print('No data available or invalid response structure.')
            
     
    

From here down, I'm working with the functions I copied over to the Pycharm project:

In [3]:
import sys
print(sys.executable)

/home/peter/data_projects/.venv/bin/python


In [5]:
from bcn_etl.functions import *
from bcn_etl.sql_queries import *
import pandas as pd
import inspect
import glob
import psycopg2

In [1]:
!uv pip freeze > requirements.txt

Using Python 3.12.3 environment at: /usr


In [ ]:
source_code = inspect.getsource(save_to_csv)

Test data for queries:

In [ ]:
resource_tuple = (2014, "f99571fd-2e6d-43f5-9eac-af4542de8aa0")
endpoint = "https://opendata-ajuntament.barcelona.cat/data/api/3/action/datastore_search"
directory = '/home/peter/data_projects/barcelona/people/'

In [ ]:
data = get_data(endpoint, resource_tuple)

In [ ]:
df = pd.read_csv('/home/peter/data_projects/barcelona/resource_ids.csv', header=None)
df.head()

In [ ]:
for row in df.itertuples(index=False):
    data = get_data(endpoint, row)
    save_to_csv(data, directory, row)
    

In [ ]:
ids = get_resource_ids('pad_mdba_sexe_edat-1')

In [ ]:
print(ids)

The following cells:
1. Collect the names of the CSV files
2. Concatenate the files into a single dataframe
3. Clean up that dataframe by changing the column names, adding a `year` column, etc.
4. Save the dataframe to a combined CSV
5. Open a connection to the database
6. Create a table for the data
7. Load the combined CSV file into the database
8. Close the connection


In [2]:
file_list = glob.glob('/home/peter/data_projects/barcelona/people/names_*.csv')

In [3]:
df = pd.concat(map(pd.read_csv, file_list), ignore_index=True)


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72001 entries, 0 to 72000
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   NOM              72001 non-null  object 
 1   SEXE             72001 non-null  int64  
 2   Valor            72001 non-null  float64
 3   Data_Referencia  72001 non-null  object 
 4   _id              72001 non-null  int64  
 5   Valor_Freq       72001 non-null  int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 3.3+ MB


In [5]:
df = df.rename(columns={'NOM': 'name', 
                        'SEXE': 'sex', 
                        'Valor': 'value', 
                        'Data_Referencia': 'reference_date',
                        'Valor_Freq': 'frequency'
                       })

In [6]:
df['reference_date'] = pd.to_datetime(df['reference_date'])
df['year'] = df['reference_date'].dt.year
df = df[['name', 'sex', 'value', 'frequency', 'year']]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72001 entries, 0 to 72000
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   name       72001 non-null  object 
 1   sex        72001 non-null  int64  
 2   value      72001 non-null  float64
 3   frequency  72001 non-null  int64  
 4   year       72001 non-null  int32  
dtypes: float64(1), int32(1), int64(2), object(1)
memory usage: 2.5+ MB


In [7]:
df.columns

Index(['name', 'sex', 'value', 'frequency', 'year'], dtype='object')

In [14]:
df.to_csv('people/names_all.csv', index=False)

In [22]:
db_configs = {
    'dbname': 'play_db',
    'user': 'player',
    'password': 'higgybig',
    'host': 'hpmini',
    'port': 5432}
conn = psycopg2.connect(**db_configs)
cur = conn.cursor()


In [23]:
columns = {'name': 'VARCHAR', 'sex': 'INT', 'value': 'REAL', 'frequency': 'INT', 'year': 'INT'}
table_name = 'bcn_names'
create_table_query = create_table(table_name, columns)
print(create_table_query)

CREATE TABLE bcn_names (name VARCHAR, sex INT, value REAL, frequency INT, year INT);


In [24]:
cur.execute(create_table_query)
conn.commit()

In [25]:
with open('people/names_all.csv', 'r') as f:
    next(f)
    cur.copy_expert(
    """
        COPY bcn_names(name, sex, value, frequency, year)
        FROM STDIN WITH CSV
        """, f
    )

In [26]:
conn.commit()

In [27]:
cur.close()
conn.close()